# Business Entity Resolution: Exploratory Data Analysis

A reproducible, lightweight audit of only the files supplied in `data/`. It uses Python's standard library, reads TSV files explicitly, and does not look up external business information.

In [ ]:
from __future__ import annotations
import csv, re, unicodedata
from collections import Counter, defaultdict
from pathlib import Path
from statistics import mean, median

ROOT = Path.cwd().resolve()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
DATA, TRAIN, TEST = ROOT / 'data', ROOT / 'data' / 'train', ROOT / 'data' / 'test'
assert TRAIN.exists() and TEST.exists(), f'Expected data/train and data/test beneath {ROOT}'
SOURCE_PATHS = {split: {f'S{i}': directory / f'{split}_source{i}.tsv' for i in (1, 2, 3)} for split, directory in {'train': TRAIN, 'test': TEST}.items()}
GROUND_TRUTH_PATH = TRAIN / 'train_ground_truth.tsv'

def read_tsv(path):
    with path.open('r', encoding='utf-8-sig', newline='') as handle:
        return list(csv.DictReader(handle, delimiter='\t'))

def headers(path):
    with path.open('r', encoding='utf-8-sig', newline='') as handle:
        return next(csv.reader(handle, delimiter='\t'), [])

def clean(value): return '' if value is None else str(value).strip()
def pct(n, d): return round(100 * n / d, 2) if d else 0.0
def present(rows, choices):
    available = set(rows[0]) if rows else set()
    return next((choice for choice in choices if choice in available), None)
def norm(value):
    value = unicodedata.normalize('NFKC', clean(value)).casefold().replace('&', ' and ')
    return re.sub(r'[^\w]+', ' ', value).strip()
def lengths(values):
    values = [len(clean(v)) for v in values if clean(v)]
    if not values: return {'non_empty': 0, 'min': None, 'p50': None, 'mean': None, 'max': None, 'bins': {}}
    bins = Counter(f'{n // 10 * 10}-{n // 10 * 10 + 9}' for n in values)
    return {'non_empty': len(values), 'min': min(values), 'p50': median(values), 'mean': round(mean(values), 2), 'max': max(values), 'bins': dict(sorted(bins.items()))}


In [ ]:
# Source-level profiles: shape, string storage type, missingness, uniqueness, duplicates, country, and length distributions.
datasets = {split: {source: read_tsv(path) for source, path in paths.items()} for split, paths in SOURCE_PATHS.items()}
ground_truth = read_tsv(GROUND_TRUTH_PATH)
for split, sources in datasets.items():
    print(f'\n{split.upper()} SOURCES')
    for source, rows in sources.items():
        fields = headers(SOURCE_PATHS[split][source])
        print(f'\n{source}: shape=({len(rows)}, {len(fields)}), columns={fields}')
        if not rows:
            print('  Header only: all values, types, missing percentages, and distributions are unobservable.')
            continue
        for field in fields:
            values = [clean(row.get(field)) for row in rows]
            print(f'  {field}: type=str, missing={pct(sum(not v for v in values), len(rows))}%, unique_nonempty={len({v for v in values if v})}')
        for label, field in [('name', present(rows, ['name', 'business_name'])), ('address', present(rows, ['address', 'business_address']))]:
            if field:
                values = [clean(row.get(field)) for row in rows]
                duplicate_rows = sum(count for value, count in Counter(values).items() if value and count > 1)
                print(f'  duplicate non-empty {label} rows={duplicate_rows} ({pct(duplicate_rows, len(rows))}%); length distribution={lengths(values)}')
        if 'country' in fields:
            countries = Counter(clean(row.get('country')) or '<missing>' for row in rows)
            print('  country distribution:', dict(countries.most_common()))

print(f'\nGROUND TRUTH: shape=({len(ground_truth)}, {len(headers(GROUND_TRUTH_PATH))}), columns={headers(GROUND_TRUTH_PATH)}')
print('Ground-truth values are read as strings to preserve identifiers and empty lists.')

In [ ]:
# Match cardinality: implemented only for the documented source1_entity_id/matched_entity_ids ground-truth contract.
def match_counts_from_documented_ground_truth(rows, source1_ids):
    if not rows or not {'source1_entity_id', 'matched_entity_ids'}.issubset(rows[0]):
        return None
    counts = {entity_id: 0 for entity_id in source1_ids}
    for row in rows:
        entity_id = clean(row['source1_entity_id'])
        matched = {item.strip() for item in clean(row['matched_entity_ids']).split(',') if item.strip()}
        if entity_id in counts: counts[entity_id] = len(matched)
    return counts

source1_ids = [clean(row.get('entity_id')) for row in datasets['train']['S1'] if clean(row.get('entity_id'))]
match_counts = match_counts_from_documented_ground_truth(ground_truth, source1_ids)
if match_counts is None:
    print('Cannot calculate Source 1 match counts: observed ground-truth schema differs from the documented contract.')
    print('Observed:', headers(GROUND_TRUTH_PATH))
elif not source1_ids:
    print('No Source 1 training records are available.')
else:
    distribution = Counter(match_counts.values())
    zero, one, multiple = (sum(n == 0 for n in match_counts.values()), sum(n == 1 for n in match_counts.values()), sum(n > 1 for n in match_counts.values()))
    ids = [item for row in ground_truth for item in clean(row['matched_entity_ids']).split(',') if item]
    print('Match-count distribution:', dict(sorted(distribution.items())))
    print('Zero/one/multiple:', {'zero': (zero, pct(zero, len(match_counts))), 'one': (one, pct(one, len(match_counts))), 'multiple': (multiple, pct(multiple, len(match_counts)))})
    print('S2/S3 match IDs:', {'S2': sum(x.startswith('S2-') for x in ids), 'S3': sum(x.startswith('S3-') for x in ids), 'other': sum(not x.startswith(('S2-', 'S3-')) for x in ids)})

In [ ]:
# Data-quality flags and candidate-blocking-key diagnostics. The keys are analysis only; no model is trained here.
def issues(rows):
    report = {}
    for label, field in [('name', present(rows, ['name', 'business_name'])), ('address', present(rows, ['address', 'business_address']))]:
        if not field: continue
        values = [clean(row.get(field)) for row in rows if clean(row.get(field))]
        report[label] = {'raw_unique': len(set(values)), 'normalized_unique': len({norm(v) for v in values}), 'normalization_collisions': len(set(values)) - len({norm(v) for v in values}), 'punctuation_or_symbol': sum(bool(re.search(r'[^\w\s]', v)) for v in values), 'non_ascii': sum(not v.isascii() for v in values)}
    return report
def block_key(row, kind):
    name, address, country, city = norm(row.get('name') or row.get('business_name')), norm(row.get('address') or row.get('business_address')), norm(row.get('country')), norm(row.get('city'))
    if kind == 'country_name_prefix': return (country, name[:8]) if name else None
    if kind == 'name_first_token': return (country, name.split()[0]) if name.split() else None
    number = next((token for token in address.split() if token.isdigit()), '')
    return (country, city, number) if number else None

all_train = [row for source in ('S1', 'S2', 'S3') for row in datasets['train'][source]]
print('Potential normalization issues:', issues(all_train))
for kind in ('country_name_prefix', 'name_first_token', 'address_number_city'):
    index = defaultdict(set)
    for source in ('S2', 'S3'):
        for row in datasets['train'][source]:
            key = block_key(row, kind)
            if key: index[key].add(clean(row.get('entity_id')))
    counts = [len(index.get(block_key(row, kind), set())) for row in datasets['train']['S1']]
    print(f'{kind}: S1 candidate coverage={pct(sum(n > 0 for n in counts), len(counts))}%, mean candidates={round(mean(counts), 2) if counts else 0}')

if not all_train:
    print('DATA BLOCKER: all source files are header-only. Populate data/ and rerun before drawing ER conclusions.')
if ground_truth and not {'source1_entity_id', 'matched_entity_ids'}.issubset(ground_truth[0]):
    print('SCHEMA BLOCKER: reconcile the ground-truth schema before label-based analysis or modeling.')
